In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content/drive/MyDrive/

API


In [6]:
import pandas as pd
import glob

# Caminhos das pastas
caminho_dados = '/content/drive/MyDrive/API/DadosBrutos/'
caminho_saida = '/content/drive/MyDrive/API/DadosLimpos/'

# Listar os arquivos CSV de exportação
arquivos_csv = sorted(glob.glob(caminho_dados + 'EXP_*.csv'))

dataframes = []

# Definir as colunas obrigatórias
colunas_obrigatorias = ['CO_ANO', 'CO_MES', 'CO_NCM', 'CO_UNID', 'CO_PAIS', 'SG_UF_NCM', 'CO_VIA', 'CO_URF', 'QT_ESTAT', 'KG_LIQUIDO', 'VL_FOB']

# Lista de códigos indesejados para CO_VIA
codigos_remover = {'00', '09', '10', '99', '12', '11', '05'}

for arquivo in arquivos_csv:
    print(f"Processando: {arquivo}")

    try:
        df = pd.read_csv(arquivo, sep=';', dtype=str)

        if all(coluna in df.columns for coluna in colunas_obrigatorias):
            df['CO_ANO'] = df['CO_ANO'].astype(int)
            df['KG_LIQUIDO'] = pd.to_numeric(df['KG_LIQUIDO'], errors='coerce')

            # Remover linhas com valores NaN
            df = df.dropna(how='all')
            df = df.dropna(subset=colunas_obrigatorias, how='any')

            # Remover linhas onde CO_VIA contém códigos indesejados
            df = df[~df['CO_VIA'].isin(codigos_remover)]

            dataframes.append(df)
        else:
            print(f"Aviso: Colunas obrigatórias ausentes em {arquivo}. Ignorando este arquivo.")

    except Exception as e:
        print(f"Erro ao processar o arquivo {arquivo}: {e}")

if dataframes:
    tabelaExp = pd.concat(dataframes, ignore_index=True)

    # Aplicar filtro de outliers na coluna KG_LIQUIDO
    Q1 = tabelaExp['KG_LIQUIDO'].quantile(0.25)
    Q3 = tabelaExp['KG_LIQUIDO'].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    tabelaExp = tabelaExp[(tabelaExp['KG_LIQUIDO'] >= limite_inferior) & (tabelaExp['KG_LIQUIDO'] <= limite_superior)]

    # Salvar o arquivo limpo
    caminho_final = caminho_saida + 'exportacao_limpa.csv'
    tabelaExp.to_csv(caminho_final, sep=';', index=False)

    print(f"Processamento concluído! Arquivo salvo como {caminho_final}")
else:
    print("Nenhum DataFrame foi carregado. Verifique os arquivos CSV.")


Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2014.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2015.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2016.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2017.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2018.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2019.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2020.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2021.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2022.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2023.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/EXP_2024.csv
Processamento concluído! Arquivo salvo como /content/drive/MyDrive/API/DadosLimpos/exportacao_limpa.csv


In [7]:
import pandas as pd
import glob

# Caminhos das pastas
caminho_dados = '/content/drive/MyDrive/API/DadosBrutos/'
caminho_saida = '/content/drive/MyDrive/API/DadosLimpos/'

# Listar os arquivos CSV de importação
arquivos_importacao = sorted(glob.glob(caminho_dados + 'IMP_*.csv'))

def processar_arquivos(lista_arquivos, tipo, caminho_saida):
    dataframes = []
    colunas_obrigatorias = ['CO_ANO', 'CO_MES', 'CO_NCM', 'CO_VIA', 'KG_LIQUIDO', 'VL_FOB', 'VL_FRETE', 'VL_SEGURO']

    # Lista de códigos indesejados para CO_VIA
    codigos_remover = {'00', '09', '10', '99', '12', '11', '05'}

    for arquivo in lista_arquivos:
        print(f"Processando: {arquivo}")
        try:
            df = pd.read_csv(arquivo, sep=';', dtype=str, encoding='utf-8')

            if all(col in df.columns for col in colunas_obrigatorias):
                for col in colunas_obrigatorias:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

                # Remover linhas com valores NaN
                df = df.dropna(how='all')
                df = df.dropna(subset=colunas_obrigatorias, how='any')

                # Remover linhas onde CO_VIA contém códigos indesejados
                df = df[~df['CO_VIA'].isin(codigos_remover)]

                # Aplicar o IQR para remover outliers
                for col in ['KG_LIQUIDO', 'VL_FOB', 'VL_FRETE', 'VL_SEGURO']:
                    Q1 = df[col].quantile(0.25)
                    Q3 = df[col].quantile(0.75)
                    IQR = Q3 - Q1
                    limite_inf = Q1 - 1.5 * IQR
                    limite_sup = Q3 + 1.5 * IQR
                    df = df[(df[col] >= limite_inf) & (df[col] <= limite_sup)]

                dataframes.append(df)
            else:
                print(f"Colunas obrigatórias ausentes em {arquivo}. Ignorando.")

        except Exception as e:
            print(f"Erro ao processar {arquivo}: {e}")

    if dataframes:
        resultado = pd.concat(dataframes, ignore_index=True)
        caminho_final = f"{caminho_saida}{tipo}_limpo.csv"
        resultado.to_csv(caminho_final, sep=';', index=False, encoding='utf-8')
        print(f"Arquivo consolidado salvo como {caminho_final}")
    else:
        print(f"Nenhum dado válido processado para {tipo}.")

processar_arquivos(arquivos_importacao, "importacao", caminho_saida)


Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2014.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2015.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2016.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2017.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2018.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2019.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2020.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2021.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2022.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2023.csv
Processando: /content/drive/MyDrive/API/DadosBrutos/IMP_2024.csv
Arquivo consolidado salvo como /content/drive/MyDrive/API/DadosLimpos/importacao_limpo.csv
